In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import requests


In [ ]:
year = 2021

In [ ]:
data_dir = Path("data")
INDIR = Path(f"../data/data_model/{year}")
OUTDIR_IMG = Path(f"../report/img/{year}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [ ]:
city_file = INDIR / f"ENEM_SCORES_MUNICIPALITIES_BRAZIL_CLUSTERS_{year}.csv"
df_city = pd.read_csv(city_file, sep=",")

In [ ]:
df_city.head()

In [ ]:
city_clusters = {
    0: df_city[df_city['CLUSTER'] == 0].copy().reset_index(drop=True),
    1: df_city[df_city['CLUSTER'] == 1].copy().reset_index(drop=True),
    2: df_city[df_city['CLUSTER'] == 2].copy().reset_index(drop=True)
}

In [ ]:
income_col = "FAMILY_INCOME_SM_AVG"

score_columns = ['NATURAL_SCIENCES_SCORE_AVG', 'HUMANITIES_SCORE_AVG', 'LANGUAGES_SCORE_AVG', 'MATH_SCORE_AVG', 'ESSAY_SCORE_AVG']

subject_names = {
    'CN': 'Natural Sciences and its Technologies',
    'MT': 'Mathematics and its Technologies',
    'CH': 'Humanities and its Technologies',
    'LC': 'Languages, Codes and its Technologies',
    'REDACAO': 'Essay'
}

score_cols = [c for c in score_columns if c in df_city.columns]
cluster_colors = {0: "#ff0e0e", 1: "#1f77b4", 2: "#2ca02c"}

performance_labels = {0: "Low", 1: "Intermediate", 2: "High"}
performance_colors = {
    "Low": cluster_colors[0],
    "Intermediate": cluster_colors[1],
    "High": cluster_colors[2],
}

In [ ]:
columns_plot = score_cols.copy()
if "OVERALL_SCORE_AVG" in df_city.columns and "OVERALL_SCORE_AVG" not in columns_plot:
    columns_plot.append("OVERALL_SCORE_AVG")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, column in enumerate(columns_plot):
    ax = axes[i]
    for cluster in sorted(df_city["CLUSTER"].unique()):
        performance = performance_labels.get(cluster, f"Cluster {cluster}")
        ax.hist(
            df_city.loc[df_city["CLUSTER"] == cluster, column],
            bins=20,
            alpha=0.55,
            label=performance,
            color=performance_colors.get(performance),
            density=True
        )

    if column == "OVERALL_SCORE_AVG":
        title = "Overall Average"
    else:
        area_code = column.replace("NOTA_", "").replace("_MEDIA", "")
        title = subject_names.get(area_code, area_code)

    ax.set_title(title)
    ax.set_xlabel("Score")
    ax.set_ylabel("Frequency")
    ax.legend()

for j in range(len(columns_plot), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Distribution of Average Scores by Cluster", fontsize=14)
plt.show()

In [ ]:
columns_plot = score_cols.copy()
if "OVERALL_SCORE_AVG" in df_city.columns and "OVERALL_SCORE_AVG" not in columns_plot:
    columns_plot.append("OVERALL_SCORE_AVG")

titles = []
for c in columns_plot:
    if c == "OVERALL_SCORE_AVG":
        area_name = "Overall Average"
    else:
        area_code = c.replace("NOTA_", "").replace("_MEDIA", "")
        area_name = subject_names.get(area_code, area_code)

    d = df_city[[income_col, c]].dropna()
    corr = d[income_col].corr(d[c])
    titles.append(f"{area_name}<br>(Correlation: {corr:.3f})")

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=titles,
    horizontal_spacing=0.08,
    vertical_spacing=0.25
)

for i, column in enumerate(columns_plot):
    row = i // 3 + 1
    col = i % 3 + 1

    for cluster in sorted(df_city["CLUSTER"].unique()):
        performance = performance_labels.get(cluster, f"Cluster {cluster}")
        data = df_city[df_city["CLUSTER"] == cluster]
        valid_data = data[[income_col, column, "CITY", "STATE", "NUM_PARTICIPANTS"]].dropna()

        fig.add_trace(
            go.Scatter(
                x=valid_data[income_col],
                y=valid_data[column],
                mode="markers",
                marker=dict(size=6, color=performance_colors.get(performance)),
                opacity=0.7,
                name=performance,
                showlegend=(i == 0),
                customdata=valid_data[["CITY", "STATE", "NUM_PARTICIPANTS"]],
                hovertemplate=(
                    "City: %{customdata[0]} (%{customdata[1]})<br>"
                    "Income: %{x:.2f}<br>"
                    "Score: %{y:.2f}<br>"
                    "Participants: %{customdata[2]}<br>"
                    f"Performance: {performance}"
                    "<extra></extra>"
                )
            ),
            row=row,
            col=col
        )

    fig.update_xaxes(title_text="Average Family Income (MW)", row=row, col=col)
    fig.update_yaxes(title_text="Average Score", row=row, col=col)

fig.update_layout(
    title=f"Relationship between Average Family Income and Scores by Performance (ENEM {year})",
    height=800,
    width=1200,
    template="plotly_white",
    legend_title_text="Performance"
)

fig.show()

In [ ]:
url = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"
geojson = requests.get(url).json()

In [ ]:
df_map = df_city.copy()

df_map["CITY_CODE"] = df_map["CITY_CODE"].astype(str)
df_map["PERFORMANCE"] = pd.Categorical(
    df_map["CLUSTER"].map(performance_labels),
    ordered=True,
    )

In [ ]:
fig = px.choropleth(
    df_map,
    geojson=geojson,
    locations="CITY_CODE",
    featureidkey="properties.id",
    color="PERFORMANCE",
    color_discrete_map=performance_colors,
    title=f"Performance by Municipality (Brazil - ENEM {year})",
    custom_data=[
        "CITY",
        "STATE",
        "PERFORMANCE",
        "OVERALL_SCORE_AVG",
        "FAMILY_INCOME_SM_AVG",
        "NUM_PARTICIPANTS"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "City: %{customdata[0]}<br>"
        "State: %{customdata[1]}<br>"
        "Performance: %{customdata[2]}<br>"
        "Overall Score: %{customdata[3]:.2f}<br>"
        "Avg. Income: %{customdata[4]:.2f}<br>"
        "Participants: %{customdata[5]}<br>"
        "<extra></extra>"
    )
)

fig.show()

In [ ]:
selected_state = "RS"

In [ ]:
df_map_state = df_map.copy()
df_map_state = df_map[df_map["STATE"] == selected_state]

df_map_state["CITY_CODE"] = df_map_state["CITY_CODE"].astype(str).str.zfill(7)

fig = px.choropleth(
    df_map_state,
    geojson=geojson,
    locations="CITY_CODE",
    featureidkey="properties.id",
    color="PERFORMANCE",
    color_discrete_map=performance_colors,
    title=f"Performance by Municipality ({selected_state} - ENEM {year})",
    custom_data=[
        "CITY",
        "STATE",
        "PERFORMANCE",
        "OVERALL_SCORE_AVG",
        "FAMILY_INCOME_SM_AVG",
        "NUM_PARTICIPANTS"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "City: %{customdata[0]}<br>"
        "State: %{customdata[1]}<br>"
        "Performance: %{customdata[2]}<br>"
        "Overall Score: %{customdata[3]:.2f}<br>"
        "Avg. Income: %{customdata[4]:.2f}<br>"
        "Participants: %{customdata[5]}<br>"
        "<extra></extra>"
    )
)

fig.show()